# Gamil RAG project

### Step 1: Import library

In [1]:
import os
import ollama
from openai import OpenAI
import json
from dotenv import load_dotenv
from huggingface_hub import login
import gradio as gr
from gmail.gmail_function import login_gmail, get_emails_list
from gmail.ingest import create_embeddings
from tqdm import tqdm
from IPython.display import display, Markdown
from pydantic import BaseModel, Field, model_validator, conint
from typing import Literal
from chromadb import PersistentClient
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient

c:\JM\projects\gmail_rag_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 2: setting variables and environment

In [2]:
load_dotenv(override = True)
ollama_api_key = os.getenv("OLLAMA_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
hf_token = os.getenv("HF_TOKEN")
credential_path = r'.credentials/credentials.json' # create credentials from Gmail API
token_path = r'.credentials/token.json' # Token will be created the first time of login to be used for subsequent login
login(hf_token, add_to_git_credential=True)

# # setup ollama
llm_client = OpenAI(base_url = "http://localhost:11434/v1/", api_key = ollama_api_key)
llm_model = "gemma3:1b"

# setup google gemini
gemini = OpenAI(base_url = "https://generativelanguage.googleapis.com/v1beta/openai/", api_key = google_api_key)
gemini_llm_model = "gemini-2.5-flash-lite"

# database and embeddings setting
DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
# setup lms
# llm_client = OpenAI(base_url = "http://localhost:1234/v1/", api_key = "lms")
# llm_model = "google/gemma-3-4b"

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Step 3: Check if gmail credentials exist

In [3]:
if not os.path.exists(credential_path):
    print(f"Error: Credentials file '{credential_path}' does not exist.")
else:
    print("Credential file exists!")

Credential file exists!


### Step 4: Login gmail with credentials or token

In [4]:
service = login_gmail(credential_path, token_path)

### Step 5: Retrieve Gmail for the past 7 days

In [6]:
emails = get_emails_list(service, days=7)

### Step 6: Create a summary for every gmail

In [7]:
class Result(BaseModel):
    page_content: str = Field(description = "email sender, date received and summary")
    metadata: dict

class Email(BaseModel):
    title: str = Field(description = "Email title")
    sender: str = Field(description = "email sender details")
    date_received: str = Field(description = "Date of email received")
    body: str = Field(description = "Email body")
    category: Literal["job", "transaction", "reminder", "project", "other"]
    summary: str = Field(description = "Email summary based on title and body")

    def as_result(self):
        metadata = {"sender": self.sender, "date_received": self.date_received, "category" : self.category}
        return Result(page_content = self.sender + " send an email on " + self.date_received + ".\nEmail summary: " + self.summary, metadata = metadata)

class Emails(BaseModel):
    emails: list[Email]

class Email_llm_output(BaseModel):
    category: Literal["job", "transaction", "reminder", "project", "other"]
    summary: str = Field(description = "Email summary based on title and body")

In [8]:
system_prompt = f"""
You will be given the title and body of an email, some emails might not have any plain text for its body.
Based on the information, respond in json format only that contains category and a summary with the structure below.
The category can only strictly be one of the five categories ["job", "transaction", "reminder", "project", "other"].
No other category is allowed.
For example,
{{
  "category": "transaction"
  "summary": string that contains summary of email title and body
}}
"""

def preprocess_emails(emails):
  
  for email in tqdm(emails):
    title = email["title"]
    body = email["body"]
    user_prompt = f""" 
    Please help to provide category and summary of the email based on title and/or body. 
    Keep the summary to one or several sentences that are clear, concise and straight to the point. 

    Here is the title of the email:
    {title}


    {"Here is the body of the email: " + body if body else "This email body does not contain any text. Please categorize and summarize based on the title only."}
    """
    messages = [{"role":"system", "content": system_prompt},
            {"role": "user", "content": user_prompt}]

    # response = llm_client.chat.completions.create(model = llm_model, messages = messages, max_tokens = 2000)
    attempt = 0
    max_attempt = 5
    while attempt < max_attempt:
      attempt += 1
      try:
        response = ollama.chat(model = llm_model, 
                          messages = messages, 
                          format=Email_llm_output.model_json_schema(),
                          options={
                                    'num_predict': 2000,  # This is Ollama's version of max_tokens
                                })
        raw_content = response["message"]["content"]
        data = Email_llm_output.model_validate(json.loads(raw_content))
        if data:
          break

      except Exception as e:
        last_exception = e
        print(f"Attempt {attempt} failed: {e}")
        if attempt == max_attempt:
          print("Max retries reached. Raising exception.")
          raise(last_exception)

    email["category"] = data.category
    email["summary"] = data.summary

  emails = Emails.model_validate({"emails":emails})
  return emails

In [9]:
emails = preprocess_emails(emails)

  9%|▉         | 6/67 [00:10<01:22,  1.36s/it]

Attempt 1 failed: Unterminated string starting at: line 1 column 30 (char 29)


 19%|█▉        | 13/67 [00:45<02:38,  2.94s/it]

Attempt 1 failed: Unterminated string starting at: line 3 column 14 (char 44)


100%|██████████| 67/67 [02:44<00:00,  2.46s/it]


### Create database based on the Email list

In [10]:
DB_PATH = "preprocessed_db"
collection_name = "docs"
create_embeddings(emails, DB_PATH = DB_PATH, collection_name = collection_name)
    

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2855.42it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore created with 67 documents


### Plot embeddings

In [11]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
categories = [metadata['category'] for metadata in metadatas]
colors = [['blue', 'green', 'yellow', 'orange', 'red'][["job", "transaction", "reminder", "project", "other"].index(category)] for category in categories]

In [12]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

### Step 8: Create RAG based on the emails

#### Retrieve email with rank order

In [13]:
RETRIEVAL_K = 10

class RankOrder(BaseModel):
    order: list[conint(ge=1, le=RETRIEVAL_K)]


In [14]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant emails from a query of a knowledge base.
The emails are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided emails by relevance to the question, with the most relevant email first.
Reply only with the list of ranked emails ids, nothing else. Include all the emails ids you are provided with, reranked.
Ensure you use each index exactly once.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the emails by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the emails:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# EMAIL ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked email ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    max_attempt = 5
    attempt = 0
    while attempt < max_attempt:
        try:
            attempt += 1
            response = ollama.chat(model=llm_model, 
                                    messages=messages, 
                                    format=RankOrder.model_json_schema())
            reply = response["message"]["content"]
            order = RankOrder.model_validate(json.loads(reply)).order
            if order:
                break
            
        except Exception as e:
            last_exception = e
            print(f"Attempt {attempt} failed: {e}")
            if attempt == max_attempt:
                print("Max retries reached. Raising exception.")
                raise(last_exception)

    return [chunks[i - 1] for i in order]

In [15]:
question = "What are the companies that offer artificial intelligence job?"


In [ ]:
# print(len(emails.emails))
# print(len(rerank_emails))
# print([email.category for email in emails.emails])
# print([email.category for email in rerank_emails])

In [16]:
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embedding_client = SentenceTransformer(embedding_model)

def fetch_context_unranked(question):
    query = embedding_client.encode(question)

    chroma = PersistentClient(path = DB_PATH)
    collection = chroma.get_or_create_collection(collection_name)
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7231.92it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
chunks = fetch_context_unranked(question)

In [18]:
for chunk in chunks:
    print(chunk.metadata["category"])

job
job
job
job
job
job
transaction
job
job
job


In [19]:
rerank_emails = rerank(question, chunks)

In [20]:
for email in rerank_emails:
    print(email.metadata["category"])

job
job
job
job
job
job
transaction
job
job
job


In [21]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

#### Retrieve email based on unranked order

In [22]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly gmail assistant.
You are chatting with a user about their gmail context.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [23]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['category']} category:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [24]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the user gmail information.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = ollama.chat(model=llm_model, messages=[{"role": "system", "content": message}])
    return response["message"]["content"]

In [25]:
rewrite_query(question, [])

'What types of AI roles are commonly offered?'

In [48]:

def normalize_message_content(content):
    if isinstance(content, list):
        return " ".join(
            item.get("text", "") for item in content if item.get("type") == "text"
        )
    return content

def answer_question(question: str, history: list[dict] = []) -> [str]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """

    history = [{"role": h["role"], "content": normalize_message_content(h["content"])} for h in history]
    print(history)
    query = rewrite_query(question, history)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = ollama.chat(model=llm_model, messages=messages)
    return response["message"]["content"]

In [44]:
answer_question(question,[])

'Okay, based on the provided context and your requests, here’s a breakdown of companies that are known to offer AI job opportunities. I’ve categorized them slightly for clarity:\n\n**Large Tech Companies with Significant AI Focus:**\n\n*   **Google:** (AI Studio, DeepMind, etc.) – Extensive AI research and applications across various products.\n*   **Microsoft:** (Azure AI, Copilot, etc.) –  AI-powered tools and services for enterprise and consumer products.\n*   **Amazon:** (AWS AI, Alexa, etc.) – Wide adoption of AI in cloud computing and services.\n*   **Meta (Facebook):**  AI for social media, metaverse, and advertising.\n*   **Apple:**  AI in product development and machine learning.\n\n**AI Focused Startups & Companies:**\n\n*   **DeepMind:** A leading AI research and development company.\n*   **Hugging Face:** Focused on open-source AI, including models and tools.\n*   **OpenAI:** Develops AI models like GPT, and has a strong focus on research and applications.\n*   **C3.ai:** F

In [33]:
answer_question("Any job related to AI",[])

'Okay, I’ve analyzed the provided context and I can answer your question about job-related AI.\n\n**Yes, there are several job roles related to AI.**\n\nHere’s a breakdown of some of the most prominent ones, based on the information provided:\n\n*   **AI Engineer:** As highlighted in the Extract from job category: LinkedIn Job Alerts, AI Engineers are responsible for developing and maintaining AI models and systems.\n*   **Machine Learning Engineer:**  This role focuses on building and deploying machine learning models.\n*   **Data Scientist:**  They analyze data, identify trends, and use it to train and improve AI models.\n*   **Research Scientist:** More involved in advancing AI technology, often leading projects and publications.\n*   **AI Specialist/Analyst:** These roles involve evaluating AI solutions, identifying opportunities for improvement, and ensuring the AI system is meeting its goals.\n*   **AI/ML Product Manager:** Focuses on defining and managing AI-driven products and 

In [49]:
gr.ChatInterface(fn=answer_question).launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


[]
[{'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': 'Hi there! How can I help you today? 😊'}]
[{'role': 'user', 'content': 'hi'}, {'role': 'assistant', 'content': 'Hi there! How can I help you today? 😊'}, {'role': 'user', 'content': 'any job related to AI in the email?'}, {'role': 'assistant', 'content': 'Yes, absolutely! I’ve identified several job-related emails focused on AI, including:\n\n*   **Artificial Intelligence Engineer - AMD Machine-Learning and Data Analytics:** This is a prominent role mentioned in the email.\n*   **Job Alert - Artificial Intelligence Engineer at AMD Machine-Learning and Data Analytics:** This highlights a specific position.\n*   **Software Engineer - Machine Learning & Data Analytics:** This email refers to a software engineering position within an AI-focused team.\n*   **Reliability Engineer, AMD | Machine-Learning and Data Analytics:** This suggests a role related to ensuring the stability and performance of AI systems.\n\nWould you

In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        gr.ChatInterface(fn=answer_question)
        
ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\JM\projects\gmail_rag_project\.venv\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\JM\projects\gmail_rag_project\.venv\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\JM\projects\gmail_rag_project\.venv\Lib\site-packages\gradio\blocks.py", line 2158, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\JM\projects\gmail_rag_project\.venv\Lib\site-packages\gradio\blocks.py", line 1632, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\JM\projects\gmail_rag_project\.venv\Lib\site-packages\gradio\utils.py", line 1007, in async_wrapper
    response = await f(*args, **k